## Node-style hooks(节点风格钩子)
顾名思义，它们在流程的 特定节点 运行。

适合顺序逻辑，如记录日志、验证。
包括
- before_agent：在Agent开始运行之前执行。
- before_model：在模型调用之前执行。
- after_model：在模型调用之后执行。
- after_agent：在Agent流程全部完成后执行。

支持两种用法
- 装饰器是函数式挂载，把一个hook快速挂载到Agent的某个节点。
- 类写法是对象化中间件，把中间件封装为一个可配置、可复用、可扩展的组件。

### 1.基本用法
#### 1.1 基于装饰器实现

In [8]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

out_model = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

In [3]:
from langchain.agents.middleware import (
    AgentMiddleware,
    AgentState,
    before_agent,
    after_agent,
    before_model,
    after_model
)
from langchain.messages import HumanMessage
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from typing import Any

#1. 定义 before_model 钩子
@before_model
def before_model_middleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += " -> before_model <- "
    return None

#2. 定义after_model 钩子
@after_model
def after_model_middleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += " -> after_model <- "
    return None

#3. 定义before_agent 钩子
@before_agent
def before_agent_middleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += " -> before_agent <- "
    return None

# 4. 定义after_agent 钩子
@after_agent
def after_agent_middleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += " -> after_agent <- "
    return None

agent = create_agent(
    model=out_model,
    # tools=[],
    middleware=[
        before_model_middleware,
        after_model_middleware,
        before_agent_middleware,
        after_agent_middleware,
    ]   # 添加自定义的中间件
)

resp = agent.invoke({
    "messages": [
        HumanMessage("你好啊")
    ]
})

for msg in resp["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好啊 -> before_agent <-  -> before_model <- 
================================== Ai Message ==================================

你好！看起来你贴了一段 AI Agent 系统的请求追踪（Trace）日志片段。按照常见的流水线设计，这条路径通常表示：

```
用户输入("你好啊") 
  → before_agent（Agent 接收前的预处理：路由、鉴权、格式校验、上下文加载等）
  → [Agent 核心逻辑：任务分解、工具调用、记忆检索等]
  → before_model（组装最终 Prompt、设置 temperature/top_p 等参数前）
  → 模型推理 → 生成回复
```

中间的 ` <- ` 可能是日志工具的流向标记或回调节点标识，实际数据流仍是单向向下的。

🔍 **你需要我协助哪部分？**
- 排查某一步骤为什么没触发或返回了空值？
- 用特定框架（LangChain / AutoGen / Dify / FastAPI + vLLM 等）实现 `before_agent` / `before_model` 的 Hook？
- 优化 Agent 与 Model 之间的参数传递或 Prompt 拼装逻辑？

告诉我你的技术栈或具体报错现象，我会给出可直接运行的代码或架构建议。 -> after_model <-  -> after_agent <-


1. 观察HumanMessage可以发现，before_agent钩子先于before_model被执行，二者都在调用模型之前被执行
2. after_agent晚于after_model执行，二者都在模型调用后执行。

#### 1.2. 基于类实现
关键规则：
1. 必须继承 AgentMiddleware ← 这个固定
2. 方法名固定 ( before_model , after_model ) ← 这个固定
3. 类名随意 ← 这个不固定

LangGraph 只看：
- 是否继承 AgentMiddleware？
- 是否有 before_model / after_model 等方法？

In [5]:
from langchain.agents.middleware import AgentMiddleware, AgentState,hook_config
from langchain.agents.middleware.types import StateT
from langchain.messages import HumanMessage
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from typing import Any

class MyMiddleware(AgentMiddleware):
    def __init__(self):
        super().__init__()

    def before_model(self, state: StateT, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content += " -> before_model <- "
        return None

    def after_model(self, state: StateT, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content += " -> after_model <- "
        return None

    def before_agent(self, state: StateT, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content += " -> before_agent <- "
        return None

    def after_agent(self, state: StateT, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content += " -> after_agent <- "
        return None

my_middleware = MyMiddleware()

agent = create_agent(
    model=out_model,
    middleware=[my_middleware],
)

resp = agent.invoke({
    "messages": [
        HumanMessage(content="你好啊")
    ]
})

for msg in resp["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好啊 -> before_agent <-  -> before_model <- 
================================== Ai Message ==================================

你好！👋  
你贴出的 `你好啊 -> before_agent <- -> before_model <-` 看起来像是一条 **AI Agent / LLM 调用链路的调试或 Hook 日志**。这种标记方式常见于自动化代理框架（如 LangChain、AutoGen、Dify、Coze 或自研 pipeline），用来追踪请求在各个节点的生命周期：

| 节点 | 典型作用 |
|------|----------|
| `before_agent` | 请求进入 Agent 编排层前：路由判断、权限校验、上下文初始化、工具注册等 |
| `before_model` | 进入底层大模型前：Prompt 拼接、参数格式化、缓存检查、Token 估算、安全过滤等 |

💡 **你想了解哪方面？**
- 这套日志是来自哪个框架/平台？（我可以帮你对照官方文档解释具体行为）
- 是否需要优化某个节点的延迟/准确率？
- 还是在开发自定义 Hook，想确认标准接入点？

提供一点背景，我就能给出更精准的链路分析或代码示例。随时告诉我～ 😊 -> after_model <-  -> after_agent <-


（1）before_model 通常的场景：
- 消息修剪（trim messages）
- PII 脱敏
- 输入验证
- 条件路由

（2）after_model 通常的场景：
- 输出验证
- 格式化响应
- 统计信息
- 状态更新

### 2. 装饰器参数：can_jump_to

#### 2.1. 基于装饰器实现

In [11]:
from typing import Any
from langchain.agents import create_agent
from langchain.agents.middleware import AgentState,before_model,after_model
from langchain.messages import AIMessage, SystemMessage
from langchain.tools import tool
from langgraph.runtime import Runtime

@tool(name_or_callable="get_news_tool")
def get_news() -> str:
    """获取当日新闻"""
    return f"美加墨世界杯今天开幕"

# 在模型（LLM）执行前触发，允许跳转到 "tools" 节点。
@before_model(can_jump_to=["tools"])
def force_tool_first(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """
    【业务场景：强行拦截并触发工具】
    如果用户输入包含 "direct tool"，则跳过本次大模型的思考/生成阶段，
    直接伪造一个大模型的 tool_calls 意图，强行把控制权移交给工具执行节点。
    """
    text = state["messages"][-1].content
    # 检测关键词，满足条件则强行干预流程
    if isinstance(text,str) and "direct tool" in text.lower():
        print("[MIDDLEWARE] before_model: jump_to='tools'")

        # 人工构造一个大模型的消息对象（AIMessage）
        # 欺骗系统，让系统误以为这是模型自己决定要调用的工具
        fake_tool_call = AIMessage(
            content="人工构造的消息",
            tool_calls=[
                {
                    "name": "get_news_tool",
                    "args": {},
                    "id": "call_fore_weather_001"
                }
            ]
        )

        # 返回更新后的状态：注入伪造的消息，并明确指定下一步跳转到 “tools” 节点
        return {
            "messages": [fake_tool_call],
            "jump_to": "tools",
        }
    # 如果不满足触发条件，返回None，流程正常向下流转（继续让LLM思考）
    return None

# 在模型（LLM）执行生成之后触发。允许重新跳转回 “model”节点。
@after_model(can_jump_to=["model"])
def retry_with_extra_instruction(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """
    【业务场景：反思/重试机制】
    如果大模型已经生成了回答，但发现用户最初的请求包含 “retry model”,
    则动态追加一条系统提示词（SystemMessage），强行让模型重新生成（重试）一次。
    """
    # 倒序遍历消息历史，找到最近的一次用户输入（human消息）
    user_text = ""
    for msg in reversed(state["messages"]):
        if getattr(msg, "type", "") == "human":
            user_text = getattr(msg, "content", "")
            break
    # 检查用户输入是否包含触发重试的关键字
    if isinstance(user_text, str) and "retry model" in user_text.lower():
        # 【核心防御】：防止无限玄幻重跳（死循环）
        # 检查消息历史中是否已经注入过这条特殊的系统提示。如果有，说明已经重试过了，不再重复干预。
        already_injected = any(
            isinstance(getattr(msg, "content", None), str)
            and "你必须以【二次回答】开头" in msg.content
            for msg in state["messages"]
        )
        if already_injected:
            return None # 已经注入过，直接放行，结束重试流程。

        print("[MIDDLEWARE] after_model: jump_to='model' with extra system instruction")

        # 返回更新后的状态：追加强力约束的系统消息，并将指针跳回 “model” 节点重新执行
        return {
            "messages": [
                SystemMessage("你必须以【二次回答】开头，并且只用一句话回答。")
            ],
            "jump_to": "model",
        }
    return None

@before_model(can_jump_to=["end"])
def overflow_context_processor(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """
    【业务场景：安全卫士/异常拦截】
    模拟上下文窗口溢出（Token超额）或其他严重的系统阻断情况。
    一旦触发，直接熔断流程，拒绝让大模型继续处理，直接报错或返回兜底方案。
    """
    # 假装溢出，模拟检查最后一条消息是否包含 overflow 标识
    if "overflow" in state["messages"][-1].content:
        print("[MIDDLEWARE] before_model: jump_to='end' when context window overflow")
        # 构造兜底的结束消息，并直接指定跳转到 "end" 终止 Agent 运行
        return {
            "messages": [
                AIMessage("上下文窗口溢出，终止")
            ],
            "jump_to": "end",
        }

agent = create_agent(
    model=out_model,
    tools=[get_news],
    # 将定义的中间件按照顺序挂载到 Agent 中（注意：执行顺序会严格按照列表声明顺序）
    middleware=[
        force_tool_first, retry_with_extra_instruction, overflow_context_processor
    ],
)

def run_once(user_input: str):
    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": user_input
                }
            ]
        }
    )

    for msg in result["messages"]:
        msg.pretty_print()

if __name__ == "__main__":
    # Case 1: 直接跳 tools
    # 预期表现：
    # 1. 触发 force_tool_first，打印 "[MIDDLEWARE] before_model: jump_to='tools'"
    # 2. 绕过 LLM 的首轮思考，直接调用 `get_news` 工具
    # 3. 工具返回结果后，LLM 总结工具结果并输出
    print('=' * 30, '-> Case 1 <-', '=' * 30)
    run_once("请帮我查今日新闻 direct tool")
    # Case 2: 输出后跳回 model
    # 预期表现：
    # 1. 正常进入 LLM 生成第 1 版回答
    # 2. 触发 retry_with_extra_instruction，打印 "[MIDDLEWARE] after_model: jump_to='model'..."
    # 3. 注入系统提示词后，LLM 被强行拉回并生成第 2 版回答
    # 4. 最终输出应带有“【二次回答】”前缀
    print('=' * 30, '-> Case 2 <-', '=' * 30)
    run_once("请随便介绍一下 LangChain retry model")
    # Case 3:
    # 预期表现：
    # 1. 触发 overflow_context_processor 中间件
    # 2. 直接打印终止信息并退出，LLM 根本不会接收到这个请求
    print('=' * 30, '-> Case 3 <-', '=' * 30)
    run_once("你好 overflow")
    # Case 4: 正常流程
    # 预期表现：
    # 1. 没有任何中间件被触发（不满足任何关键字）
    # 2. Agent 走正常的 OOTB（Out of the box）标准工作流：User -> Model -> Call Tool -> Model -> End
    print('=' * 30, '-> Case 4 <-', '=' * 30)
    run_once("今日新闻摘要？")

============================== -> Case 1 <- ==============================
[MIDDLEWARE] before_model: jump_to='tools'
================================ Human Message =================================

请帮我查今日新闻 direct tool
================================== Ai Message ==================================

人工构造的消息
Tool Calls:
  get_news_tool (call_fore_weather_001)
 Call ID: call_fore_weather_001
  Args:
================================= Tool Message =================================
Name: get_news_tool

美加墨世界杯今天开幕
================================== Ai Message ==================================

今日新闻：美加墨世界杯今天开幕。
============================== -> Case 2 <- ==============================
[MIDDLEWARE] after_model: jump_to='model' with extra system instruction
================================ Human Message =================================

请随便介绍一下 LangChain retry model
================================== Ai Message ==================================

其实 LangChain 里并没有一个叫 **“retry model”** 的独立组件

#### 2.2. 基于类实现

In [12]:
from typing import Any
from langchain.agents import create_agent
from langchain.agents.middleware import hook_config, AgentState, AgentMiddleware
from langchain.messages import AIMessage, SystemMessage
from langchain.tools import tool
from langgraph.runtime import Runtime

@tool(name_or_callable="get_news_tool")
def get_news() -> str:
    """获取当日新闻"""
    return f"美加墨世界杯今天开幕"

class MyMiddleware(AgentMiddleware):
    @hook_config(can_jump_to=["tools", "end"])
    def before_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        text = state["messages"][-1].content
        # 假装溢出
        if "overflow" in text:
            print("[MIDDLEWARE] before_model: jump_to='end' when context window overflow")
            return {
                "messages": [
                    AIMessage(content="上下文窗口溢出，终止")
                ],
                "jump_to": "end",
            }

        if isinstance(text, str) and "direct tool" in text.lower():
            print("[MIDDLEWARE] before_model: jump_to='tools'")
            fake_tool_call = AIMessage(
                content="人工构造的消息",
                tool_calls=[
                    {
                        "name": "get_news_tool",
                        "args": {},
                        "id": "call_force_weather_001"
                    }
                ]
            )

            return {
                "messages": [fake_tool_call],
                "jump_to": "tools",
            }
        return None

    @hook_config(can_jump_to=["model"])
    def after_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        user_text = ""
        for msg in reversed(state["messages"]):
            if getattr(msg, "type", "") == "human":
                user_text = getattr(msg, "content", "")
                break
        if isinstance(user_text, str) and "retry model" in user_text.lower():
            # 防止无限重跳：如果已经加过提示，就不再跳
            already_injected = any(
                isinstance(getattr(msg, "content", None), str)
                and "你必须以【二次回答】开头" in msg.content
                for msg in state["messages"]
            )
            if already_injected:
                return None

            print("[MIDDLEWARE] after_model: jump_to='model' with extra system instruction")

            # 返回更新后的状态：追加强力约束的系统消息，并将指针跳回 “model” 节点重新执行
            return {
                "messages": [
                    SystemMessage("你必须以【二次回答】开头，并且只用一句话回答。")
                ],
                "jump_to": "model",
            }
        return None

agent = create_agent(
    model=out_model,
    tools=[get_news],
    middleware=[
        MyMiddleware()
    ],
)

def run_once(user_input: str):
    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": user_input
                }
            ]
        }
    )

    for msg in result["messages"]:
        msg.pretty_print()

if __name__ == "__main__":
    # Case 1: 直接跳 tools
    print('=' * 30, '-> Case 1 <-', '=' * 30)
    run_once("请帮我查今日新闻 direct tool")
    # Case 2: 输出后跳回 model
    print('=' * 30, '-> Case 2 <-', '=' * 30)
    run_once("请随便介绍一下 LangChain retry model")
    # Case 3:
    print('=' * 30, '-> Case 3 <-', '=' * 30)
    run_once("你好 overflow")
    # Case 4: 正常流程
    print('=' * 30, '-> Case 4 <-', '=' * 30)
    run_once("今日新闻摘要？")

============================== -> Case 1 <- ==============================
[MIDDLEWARE] before_model: jump_to='tools'
================================ Human Message =================================

请帮我查今日新闻 direct tool
================================== Ai Message ==================================

人工构造的消息
Tool Calls:
  get_news_tool (call_force_weather_001)
 Call ID: call_force_weather_001
  Args:
================================= Tool Message =================================
Name: get_news_tool

美加墨世界杯今天开幕
================================== Ai Message ==================================

今天是2026年7月15日，据消息显示，美加墨世界杯今天开幕。
============================== -> Case 2 <- ==============================
[MIDDLEWARE] after_model: jump_to='model' with extra system instruction
================================ Human Message =================================

请随便介绍一下 LangChain retry model
================================== Ai Message ==================================

LangChain 官方并没有一个叫 **“retry